# BAMv5: FFT 기반 Supervised Denoising

**핵심 혁신**
- ✅ STFT 제거 → FFT 직접 사용
- ✅ 완벽한 길이 보존 (2048 → 2048)
- ✅ Clean target (명확한 디노이징)
- ✅ On-the-fly augmentation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import importlib

# 강제 리로드
import utils.LoRa
import utils.my_lora_utils
importlib.reload(utils.LoRa)
importlib.reload(utils.my_lora_utils)

from utils.LoRa import LoRa, MultiBAMv5
from utils.my_lora_utils import *

print(f"LoRa module: {utils.LoRa.__file__}")
print(f"Utils module: {utils.my_lora_utils.__file__}")

LoRa module: /home/gwon9906/LoRa-bam-reconstruction/utils/LoRa.py
Utils module: /home/gwon9906/LoRa-bam-reconstruction/utils/my_lora_utils.py


In [2]:
########################################
##  직접 IQ 변환 함수 (Time Domain)  ##
########################################

def generate_iq_features(iq_signal):
    """
    IQ 신호를 직접 사용 (시간 도메인)
    FFT 없이 Real/Imag만 분리
    
    Parameters:
    -----------
    iq_signal : complex array
        복소 IQ 신호 (2048 samples)
    
    Returns:
    --------
    combined_flat : ndarray
        Real과 Imag를 통합한 1D 배열 (4096,)
    norm_params : dict
        역정규화를 위한 파라미터
    """
    # Real/Imag 분리
    real_part = iq_signal.real
    imag_part = iq_signal.imag
    
    # 정규화 (0-1 범위)
    r_min, r_max = real_part.min(), real_part.max()
    i_min, i_max = imag_part.min(), imag_part.max()
    
    real_norm = (real_part - r_min) / (r_max - r_min + 1e-12)
    imag_norm = (imag_part - i_min) / (i_max - i_min + 1e-12)
    
    # Concatenate
    combined_flat = np.concatenate([real_norm, imag_norm])
    
    # 역변환 정보 저장
    norm_params = {
        'r_min': r_min, 'r_max': r_max,
        'i_min': i_min, 'i_max': i_max,
        'signal_len': len(iq_signal)
    }
    
    return combined_flat, norm_params

# 별칭 (기존 코드 호환)
generate_fft_features = generate_iq_features

print("✅ 직접 IQ 변환 함수 정의 완료 (Time Domain, 시간 정보 100% 보존)")

✅ 직접 IQ 변환 함수 정의 완료 (Time Domain, 시간 정보 100% 보존)


In [3]:
# LoRa 파라미터
sf = 9
bw = 250_000
OSF = 4
fs = int(bw * OSF)

print(f"SF = {sf}, BW = {bw/1e3:.1f} kHz, fs = {fs/1e6:.3f} MHz")

lora = LoRa(sf, bw)

SF = 9, BW = 250.0 kHz, fs = 1.000 MHz


In [4]:
########################################
##  1. 클린 IQ 데이터 로드            ##
########################################

# v3 데이터셋 재사용 또는 v5 데이터셋 사용
base_dir = "../Model-experiment-v3/dataset_v3_sf9_bw250k"  # 또는 dataset_v5_sf9_bw250k
iq_dir = os.path.join(base_dir, "clean_iq")

print("Loading clean IQ data...")
iq_files = sorted(glob.glob(os.path.join(iq_dir, "*.npy")))
print(f"Found {len(iq_files)} clean IQ files")

iq_data_list = []
for f in iq_files:
    x_clean = np.load(f)
    iq_data_list.append(x_clean)

X_clean_iq = np.array(iq_data_list)
print(f"Clean IQ shape: {X_clean_iq.shape}, dtype: {X_clean_iq.dtype}")

Loading clean IQ data...
Found 512 clean IQ files
Clean IQ shape: (512, 2048), dtype: complex64


In [5]:
########################################
##  2. 데이터 증강 함수 (Clean Target)  ##
########################################

SNR_MIN = -25
SNR_MAX = -5
AUGMENT_FACTOR = 30

def augment_dataset_supervised(X_clean_iq, augment_factor=30):
    """
    Supervised learning: Noisy → Clean
    
    Returns:
        X_noisy_iq: (N, 4096) - Noisy IQ features (Time Domain)
        X_clean_iq: (N, 4096) - Clean IQ features (target)
    """
    print(f"\nGenerating {len(X_clean_iq)} × {augment_factor} samples...")
    
    X_noisy_list = []
    X_clean_list = []
    
    for clean_iq in X_clean_iq:
        # Clean IQ features (target)
        clean_features, _ = generate_iq_features(clean_iq)
        
        for _ in range(augment_factor):
            # 랜덤 SNR
            snr = np.random.randint(SNR_MIN, SNR_MAX + 1)
            
            # 노이즈 추가
            noisy_iq = lora.awgn_iq(clean_iq, snr)
            
            # Noisy IQ features (input)
            noisy_features, _ = generate_iq_features(noisy_iq)
            
            X_noisy_list.append(noisy_features)
            X_clean_list.append(clean_features)
    
    X_noisy_fft = np.array(X_noisy_list)
    X_clean_fft = np.array(X_clean_list)
    
    print(f"Noisy shape: {X_noisy_fft.shape}")
    print(f"Clean shape: {X_clean_fft.shape}")
    
    return X_noisy_fft, X_clean_fft

print(f"✅ Supervised augmentation function defined: {AUGMENT_FACTOR}x")

✅ Supervised augmentation function defined: 30x


In [6]:
########################################
##  3. 모델 아키텍처 설정             ##
########################################

# IQ features 차원 확인
test_features, test_params = generate_iq_features(X_clean_iq[0])
input_dim = test_features.shape[0]

# 부드러운 압축: 4096 → 1024 → 256 → 64 (64배 압축!)
layers = [input_dim, 1024, 256, 64]

print(f"IQ signal length: {test_params['signal_len']}")
print(f"Input dim (Real+Imag): {input_dim}")
print(f"Model architecture: {' → '.join(map(str, layers))}")
print(f"Compression ratio: {input_dim / layers[-1]:.1f}x")
print(f"Layers: {len(layers)-1} hidden layers")
print(f"✅ Time domain: 시간 정보 100% 보존!")

IQ signal length: 2048
Input dim (Real+Imag): 4096
Model architecture: 4096 → 1024 → 256 → 64
Compression ratio: 64.0x
Layers: 3 hidden layers
✅ Time domain: 시간 정보 100% 보존!


In [7]:
########################################
##  4. BAMv5 학습 (Supervised)       ##
########################################

print(f"\n=== Training BAMv5 Model (FFT + Clean Target) ===")
print(f"Architecture: {' → '.join(map(str, layers))}")
print(f"Strategy: Supervised Denoising (Noisy → Clean)")

model = MultiBAMv5(layers_dims=layers, eta=5e-4)

TRAIN = True  # ⚠️ True로 변경하여 학습

if TRAIN:
    print("\n=== Starting Supervised Training ===")
    print(f"Base samples: {len(X_clean_iq)}")
    print(f"Augmentation: {AUGMENT_FACTOR}x")
    print(f"Total per epoch: {len(X_clean_iq) * AUGMENT_FACTOR}")
    
    num_epochs = 20
    batch_size = 64
    
    all_layer_losses = [[] for _ in range(len(layers)-1)]
    
    for epoch in range(num_epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"{'='*60}")
        
        # On-the-fly augmentation
        X_noisy, X_clean = augment_dataset_supervised(X_clean_iq, AUGMENT_FACTOR)
        
        # Train with clean target
        layer_losses = model.train_supervised(X_noisy, X_clean, num_epochs=1, batch_size=batch_size)
        
        for i, losses in enumerate(layer_losses):
            all_layer_losses[i].extend(losses)
    
    print("\n=== Training Complete ===")
    
    # Loss plot
    fig, axes = plt.subplots(1, len(all_layer_losses), figsize=(15, 4))
    if len(all_layer_losses) == 1:
        axes = [axes]
    
    for i, losses in enumerate(all_layer_losses):
        axes[i].plot(losses)
        axes[i].set_xlabel('Batch')
        axes[i].set_ylabel('MSE Loss')
        axes[i].set_title(f'Layer {i+1} (Noisy→Clean)')
        axes[i].grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("\nTraining skipped (TRAIN=False)")


=== Training BAMv5 Model (FFT + Clean Target) ===
Architecture: 4096 → 1024 → 256 → 64
Strategy: Supervised Denoising (Noisy → Clean)

=== Starting Supervised Training ===
Base samples: 512
Augmentation: 30x
Total per epoch: 15360

Epoch 1/20

Generating 512 × 30 samples...
Noisy shape: (15360, 4096)
Clean shape: (15360, 4096)

--- Training Layer 1/3 ---
   Input shape: (15360, 4096)
   Target shape: (15360, 4096)
Epoch 1, Batch 1/240, MSE = 0.179518
Epoch 1, Batch 2/240, MSE = 0.170163
Epoch 1, Batch 3/240, MSE = 0.170589
Epoch 1, Batch 4/240, MSE = 0.170022
Epoch 1, Batch 5/240, MSE = 0.170130
Epoch 1, Batch 6/240, MSE = 0.169961
Epoch 1, Batch 7/240, MSE = 0.171181
Epoch 1, Batch 8/240, MSE = 0.170375
Epoch 1, Batch 9/240, MSE = 0.168426
Epoch 1, Batch 10/240, MSE = 0.168604
Epoch 1, Batch 11/240, MSE = 0.169151
Epoch 1, Batch 12/240, MSE = 0.169716
Epoch 1, Batch 13/240, MSE = 0.168940
Epoch 1, Batch 14/240, MSE = 0.169810
Epoch 1, Batch 15/240, MSE = 0.169284
Epoch 1, Batch 16/24

KeyboardInterrupt: 

In [ ]:
########################################
##  5. 모델 저장                     ##
########################################

import torch

if TRAIN:
    weight_folder = "weights_bamv5_fft"
    os.makedirs(weight_folder, exist_ok=True)
    
    for i, bam in enumerate(model.bams):
        weight_path = os.path.join(weight_folder, f"weights_layer_{i}.npy")
        np.save(weight_path, bam.W.cpu().numpy())
        print(f"✅ Layer {i} saved: {weight_path}")
    
    config = {
        'layers': layers,
        'eta': 5e-4,
        'sf': sf,
        'bw': bw,
        'fs': fs,
        'mode': 'fft_supervised'
    }
    np.save(os.path.join(weight_folder, "model_config.npy"), config)
    print(f"✅ Config saved")

print("\n=== Model Save Complete ===")

✅ Layer 0 saved: weights_bamv5_fft/weights_layer_0.npy
✅ Layer 1 saved: weights_bamv5_fft/weights_layer_1.npy
✅ Layer 2 saved: weights_bamv5_fft/weights_layer_2.npy
✅ Config saved

=== Model Save Complete ===


In [ ]:
########################################
##  5-1. 모델 로드 (TRAIN=False)     ##
########################################

import torch

if not TRAIN:
    weight_folder = "weights_bamv5_fft"
    config_path = os.path.join(weight_folder, "model_config.npy")
    
    if os.path.exists(config_path):
        print(f"\n=== Loading Pre-trained Model ===")
        
        config = np.load(config_path, allow_pickle=True).item()
        print(f"Loaded config: {config}")
        
        if config['layers'] == layers:
            for i, bam in enumerate(model.bams):
                weight_path = os.path.join(weight_folder, f"weights_layer_{i}.npy")
                if os.path.exists(weight_path):
                    loaded_weight = np.load(weight_path)
                    bam.W = torch.tensor(loaded_weight, dtype=torch.float32).to(bam.device)
                    print(f"✅ Layer {i} loaded (device: {bam.device})")
            
            print(f"✅ Model loaded from {weight_folder}")
        else:
            print(f"❌ Architecture mismatch!")
    else:
        print(f"❌ No pre-trained model found")
else:
    print("\n=== Using Newly Trained Model ===")


=== Using Newly Trained Model ===


In [ ]:
########################################
##  6. 테스트: FFT 복원 및 IQ 재구성  ##
########################################

test_symbol = 42
test_snr = -10

print(f"\n=== Testing FFT Reconstruction ===")
print(f"Test symbol: {test_symbol}, SNR: {test_snr} dB")

# 1. Clean IQ
clean_iq = X_clean_iq[test_symbol]

# 2. Noisy IQ
noisy_iq = lora.awgn_iq(clean_iq, test_snr)

# 3. IQ features 변환 (Time Domain)
noisy_features, norm_params = generate_iq_features(noisy_iq)
clean_features, clean_params = generate_iq_features(clean_iq)

# 4. BAM 복원
compressed = model.compress(noisy_features.reshape(1, -1))
restored_features = model.decompress(compressed).flatten()

print(f"\n=== Feature Domain Comparison ===")
mse_noisy_clean = ((noisy_features - clean_features)**2).mean()
mse_restored_clean = ((restored_features - clean_features)**2).mean()
improvement = (1 - mse_restored_clean/mse_noisy_clean) * 100

print(f"Noisy vs Clean:    MSE = {mse_noisy_clean:.6f}")
print(f"Restored vs Clean: MSE = {mse_restored_clean:.6f}")
print(f"Improvement: {improvement:.1f}%")

# 5. 역정규화 (IQ 신호 복원)
mid = len(restored_features) // 2
real_restored_norm = restored_features[:mid]
imag_restored_norm = restored_features[mid:]

# 역정규화
real_restored = real_restored_norm * (norm_params['r_max'] - norm_params['r_min']) + norm_params['r_min']
imag_restored = imag_restored_norm * (norm_params['i_max'] - norm_params['i_min']) + norm_params['i_min']

# 6. Complex IQ 재결합
iq_reconstructed = real_restored + 1j * imag_restored

print(f"\n✅ IQ reconstructed: {iq_reconstructed.shape}")
print(f"   Length match: {len(iq_reconstructed) == len(clean_iq)} ✅")
print(f"   No FFT/iFFT! Direct time domain reconstruction")

# 8. 에너지 정규화
energy_clean = np.sqrt(np.mean(np.abs(clean_iq)**2))
energy_reconstructed = np.sqrt(np.mean(np.abs(iq_reconstructed)**2))

if energy_reconstructed > 1e-10:
    scale_factor = energy_clean / energy_reconstructed
    iq_reconstructed = iq_reconstructed * scale_factor
    print(f"   Energy scaling: {scale_factor:.2f}x")

# 9. IQ Domain MSE
print(f"\n=== IQ Domain Comparison ===")
mse_clean_noisy = np.mean(np.abs(clean_iq - noisy_iq)**2)
mse_clean_restored = np.mean(np.abs(clean_iq - iq_reconstructed)**2)
iq_improvement = (1 - mse_clean_restored/mse_clean_noisy) * 100

print(f"Clean vs Noisy:    MSE = {mse_clean_noisy:.6f}")
print(f"Clean vs Restored: MSE = {mse_clean_restored:.6f}")
print(f"Improvement: {iq_improvement:.1f}%")


=== Testing FFT Reconstruction ===
Test symbol: 42, SNR: -10 dB

=== FFT Domain Comparison ===
Noisy vs Clean:    MSE = 0.075256
Restored vs Clean: MSE = 0.115479
Improvement: -53.4%

✅ IQ reconstructed: (2048,)
   Length match: True
   Energy scaling: 0.49x

=== IQ Domain Comparison ===
Clean vs Noisy:    MSE = 10.300019
Clean vs Restored: MSE = 1.619660
Improvement: 84.3%


In [ ]:
########################################
##  7. Dechirp 심볼 복원 평가         ##
########################################

from utils.my_lora_utils import estimate_symbol_custom

print("\n=== Dechirp Symbol Decoding ===")

code_clean, peak_clean = estimate_symbol_custom(clean_iq, "clean", sf, fs, bw)
print(f"✅ Clean:    Code {code_clean}, Peak {peak_clean:.2f}")

code_noisy, peak_noisy = estimate_symbol_custom(noisy_iq, "noisy", sf, fs, bw)
print(f"⚠️  Noisy:    Code {code_noisy}, Peak {peak_noisy:.2f}")

code_restored, peak_restored = estimate_symbol_custom(iq_reconstructed, "restored", sf, fs, bw)
print(f"🔧 Restored: Code {code_restored}, Peak {peak_restored:.2f}")

print(f"\n{'='*60}")
print(f"Performance Evaluation:")
print(f"{'='*60}")
print(f"Ground Truth: {test_symbol}")
print(f"Clean:    {code_clean} {'✅' if code_clean == test_symbol else '❌'}")
print(f"Noisy:    {code_noisy} {'✅' if code_noisy == test_symbol else '❌'}")
print(f"Restored: {code_restored} {'✅' if code_restored == test_symbol else '❌'}")
print(f"{'='*60}")

print(f"\nPeak Comparison:")
print(f"Clean:    {peak_clean:.2f}")
print(f"Noisy:    {peak_noisy:.2f} ({peak_noisy/peak_clean*100:.1f}%)")
print(f"Restored: {peak_restored:.2f} ({peak_restored/peak_clean*100:.1f}%)")

if code_restored == test_symbol:
    print(f"\n✅ SUCCESS: BAMv5 restored correct symbol!")
else:
    print(f"\n❌ FAILURE: Symbol mismatch")


=== Dechirp Symbol Decoding ===
✅ Clean:    Code 42, Peak 1880.00
⚠️  Noisy:    Code 42, Peak 1798.73
🔧 Restored: Code 15, Peak 469.02

Performance Evaluation:
Ground Truth: 42
Clean:    42 ✅
Noisy:    42 ✅
Restored: 15 ❌

Peak Comparison:
Clean:    1880.00
Noisy:    1798.73 (95.7%)
Restored: 469.02 (24.9%)

❌ FAILURE: Symbol mismatch
